# Module 1: Sentence Roles — Training Sketch

Interactive walkthrough of what `ml/scripts/train_sentence_roles.py` does:
load the processed splits, build the per-sentence dataset, train a
TF-IDF + linguistic-feature `LogisticRegression` classifier, and inspect
predictions on a few examples.

For the full, reproducible CLI version see:

```bash
cd ml
python scripts/prepare_data.py
python scripts/train_sentence_roles.py
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # make `src` importable, mirrors scripts/train_sentence_roles.py

from src.data_utils import build_role_dataset, load_split
from src.metrics import classification_metrics, print_metrics_summary
from src.models.classifiers import SklearnRoleClassifier

train = load_split("train")
val = load_split("val")
test = load_split("test")

train_texts, train_pos, train_tot, train_labels = build_role_dataset(train)
val_texts, val_pos, val_tot, val_labels = build_role_dataset(val)
test_texts, test_pos, test_tot, test_labels = build_role_dataset(test)

print(f"train sentences: {len(train_texts)}, val: {len(val_texts)}, test: {len(test_texts)}")

## Train the classifier

In [ ]:
model = SklearnRoleClassifier(max_features=3000, C=2.0)
model.fit(train_texts, train_pos, train_tot, train_labels)

test_preds = model.predict(test_texts, test_pos, test_tot)
metrics = classification_metrics(test_labels, test_preds, labels=list(SklearnRoleClassifier.ROLES))
print_metrics_summary("Sentence Roles - Test (notebook run)", metrics)

## Inspect a few predictions

Spot-check a handful of test sentences against their true vs. predicted
role, to build intuition for where the model struggles (usually `Other` vs.
a weakly-worded `Claim`, or `Evidence` vs. `Explanation` on borderline
sentences).

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "sentence": test_texts,
        "true_role": test_labels,
        "predicted_role": test_preds,
    }
)
comparison["correct"] = comparison["true_role"] == comparison["predicted_role"]
comparison.sort_values("correct").head(10)